# ESMC Mutation Scoring - Modal Version

Score mutations at scale using ESMC on Modal's serverless GPU infrastructure.

**What it does:**
- Masks each position and runs inference remotely on Modal H100 GPU
- Computes Shannon entropy (conservation at each position)
- Computes log-likelihood ratios (effect of mutations)
- Scales to 100s of proteins without local GPU

**Advantages over Colab version:**
- No GPU memory constraints
- Serverless (pay per minute, no idle billing)
- Persistent model caching
- Better for batch processing 100+ proteins


## Setup

In [ ]:
!pip install -q modal pandas numpy tqdm ipywidgets matplotlib scipy

In [ ]:
import modal
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
from IPython.display import display, Markdown
import time

print(f"Modal version: {modal.__version__}")
print("Dependencies loaded successfully")

## Modal Authentication

In [ ]:
#@markdown Get API token from https://modal.com/account/tokens

from getpass import getpass

MODAL_API_TOKEN = getpass("Modal API Token: ")

print("✓ Modal token configured")

## Define Modal App with Mutation Scorer

In [ ]:
# Modal app
app = modal.App("esmc-mutation-scoring")

ESM_REVISION = "main"
MINUTES = 60

# Persistent volume for model caching
models_dir = "/root/.cache/huggingface"
models_volume = modal.Volume.from_name(
    "esmc-models-cache",
    create_if_missing=True
)

# Container image
image = (
    modal.Image.debian_slim(python_version="3.13")
    .apt_install("git")
    .pip_install(
        "esm @ git+https://github.com/Biohub/esm.git@" + ESM_REVISION,
        "torch>=2.2.0",
        "transformers",
        "accelerate",
        "einops",
        "numpy",
        "pandas",
    )
    .env({"HF_HOME": models_dir})
)

print("✓ Modal app configured")
print("✓ Container image defined")
print("✓ Volume created for model caching")

In [ ]:
# Canonical amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

@app.cls(
    image=image,
    volumes={models_dir: models_volume},
    gpu="H100",
    timeout=30 * MINUTES,
    concurrency_limit=1,
)
class ESCMMutationScorer:
    """Remote ESMC mutation scoring on Modal H100 GPU."""
    
    @modal.enter()
    def setup(self):
        """Load model once at startup."""
        import torch
        import torch.nn.functional as F
        from esm.models import ESMCForCausalLM
        from esm.tokenization import TokenizerBase
        
        print("Loading ESMC model...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = ESMCForCausalLM.from_pretrained("esm2/esmc_600m").to(self.device)
        self.model.eval()
        self.tokenizer = TokenizerBase()
        self.F = F
        self.torch = torch
        self.np = __import__('numpy')
        print(f"✓ Model loaded on {self.device}")
    
    def _get_leave_one_out_logits(self, sequence: str) -> np.ndarray:
        """Get logits with each position masked."""
        logits_list = []
        
        # Create masked sequences
        masked_sequences = [
            sequence[:i] + "_" + sequence[i + 1:]
            for i in range(len(sequence))
        ]
        
        with self.torch.no_grad():
            for batch_start in range(0, len(masked_sequences), 4):
                batch_end = min(batch_start + 4, len(masked_sequences))
                batch_seqs = masked_sequences[batch_start:batch_end]
                
                # Tokenize
                tokens_batch = [self.tokenizer.tokenize(seq) for seq in batch_seqs]
                max_len = max(len(t) for t in tokens_batch)
                tokens_padded = [
                    t + [self.tokenizer.pad_id] * (max_len - len(t))
                    for t in tokens_batch
                ]
                tokens_tensor = self.torch.tensor(tokens_padded).to(self.device)
                
                # Forward pass
                output = self.model(tokens_tensor)
                logits = output.logits.cpu().numpy()
                
                # Extract logits at masked position
                for j, seq_idx in enumerate(range(batch_start, batch_end)):
                    pos = seq_idx + 1
                    logits_list.append(logits[j, pos, :])
        
        return self.np.array(logits_list)
    
    def _compute_entropy(self, logits: np.ndarray) -> np.ndarray:
        """Compute Shannon entropy from logits."""
        logits_torch = self.torch.from_numpy(logits).float()
        probs = self.F.softmax(logits_torch, dim=-1).numpy()
        entropy = -self.np.sum(probs * self.np.log2(probs + 1e-10), axis=-1)
        return entropy
    
    def _compute_llr(self, logits: np.ndarray, sequence: str) -> np.ndarray:
        """Compute log-likelihood ratios."""
        logits_torch = self.torch.from_numpy(logits).float()
        log_probs = self.F.log_softmax(logits_torch, dim=-1)
        
        aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
        llr_list = []
        
        for i, wt_aa in enumerate(sequence):
            if wt_aa not in aa_to_idx:
                llr_list.append(self.np.zeros(len(AMINO_ACIDS)))
                continue
            
            wt_idx = aa_to_idx[wt_aa]
            wt_logprob = log_probs[i, wt_idx].item()
            
            llr_row = []
            for aa_idx in range(len(AMINO_ACIDS)):
                aa_logprob = log_probs[i, aa_idx].item()
                llr = aa_logprob - wt_logprob
                llr_row.append(llr)
            llr_list.append(llr_row)
        
        return self.np.array(llr_list)
    
    @modal.method()
    def score_mutations(self, sequence: str, seq_id: str = "protein") -> Dict:
        """Score all mutations in a sequence.
        
        Returns:
            Dictionary with entropy, llr, and metrics
        """
        print(f"Scoring mutations for {seq_id} ({len(sequence)} aa)")
        
        # Get logits
        logits = self._get_leave_one_out_logits(sequence)
        
        # Compute metrics
        entropy = self._compute_entropy(logits)
        llr = self._compute_llr(logits, sequence)
        frac_del = self.np.mean(llr < 0, axis=1)
        
        print(f"✓ Mean entropy: {self.np.mean(entropy):.2f} bits")
        print(f"✓ Mean fraction deleterious: {self.np.mean(frac_del):.2f}")
        
        return {
            "sequence": sequence,
            "entropy": entropy.tolist(),
            "llr": llr.tolist(),
            "fraction_deleterious": frac_del.tolist(),
        }

print("✓ ESCMMutationScorer class defined")

## Configuration & Input

In [ ]:
OUTPUT_DIR = "/content/mutation_scoring_modal_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
#@markdown Upload FASTA or use examples

from google.colab import files

def load_fasta(filepath: str) -> Dict[str, str]:
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example proteins
    sequences = {
        "GFP": "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVAWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
        "ubiquitin": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in sequences.items():
    print(f"  {seq_id}: {len(seq)} aa")

## Run Remote Mutation Scoring

In [ ]:
print("\n🚀 Triggering remote mutation scoring on Modal H100 GPU...\n")

scorer = ESCMMutationScorer()
results = {}
errors = []

for seq_id, sequence in tqdm(sequences.items(), desc="Scoring mutations"):
    try:
        result = scorer.score_mutations.remote(sequence, seq_id)
        results[seq_id] = result
    except Exception as e:
        print(f"Error scoring {seq_id}: {e}")
        errors.append((seq_id, str(e)))

print(f"\n✓ Completed {len(results)}/{len(sequences)} scoring jobs")
if errors:
    print(f"✗ Failed: {len(errors)}")

## Visualization & Analysis

In [ ]:
import matplotlib.pyplot as plt

for seq_id, result in results.items():
    sequence = result["sequence"]
    entropy = np.array(result["entropy"])
    llr = np.array(result["llr"])
    frac_del = np.array(result["fraction_deleterious"])
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f"{seq_id} - Mutation Scoring Analysis", fontsize=14, fontweight='bold')
    
    # 1. Entropy
    ax = axes[0, 0]
    ax.plot(entropy, linewidth=2, color='steelblue')
    ax.fill_between(range(len(entropy)), entropy, alpha=0.3)
    ax.axhline(y=np.log2(20), color='red', linestyle='--', alpha=0.3, label='Uniform 20 AAs')
    ax.set_xlabel('Position')
    ax.set_ylabel('Entropy (bits)')
    ax.set_title('Shannon Entropy (Conservation)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Deleterious Fraction
    ax = axes[0, 1]
    ax.scatter(range(len(frac_del)), frac_del, alpha=0.6, s=30, color='coral')
    ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Position')
    ax.set_ylabel('Fraction Deleterious')
    ax.set_title('Mutation Tolerance')
    ax.set_ylim([-0.05, 1.05])
    ax.grid(True, alpha=0.3, axis='y')
    
    # 3. LLR Heatmap (top 100 features)
    ax = axes[1, 0]
    im = ax.imshow(llr.T[:, :min(100, len(llr))], aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set_xlabel('Position (first 100)')
    ax.set_ylabel('Amino Acid')
    ax.set_title('LLR Heatmap')
    plt.colorbar(im, ax=ax, label='LLR')
    
    # 4. Statistics
    ax = axes[1, 1]
    ax.axis('off')
    stats_text = f"""
    Sequence Statistics:
    
    Length: {len(sequence)} aa
    
    Entropy:
    • Mean: {np.mean(entropy):.2f} bits
    • Min: {np.min(entropy):.2f} bits
    • Max: {np.max(entropy):.2f} bits
    
    Mutation Tolerance:
    • Mean frac_del: {np.mean(frac_del):.2f}
    • Flexible pos (<0.8): {np.sum(frac_del < 0.8)}
    • Constrained (>0.9): {np.sum(frac_del > 0.9)}
    """
    ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
            verticalalignment='center')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{seq_id}_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()

print("✓ Analysis plots saved")

## Top Mutation Recommendations

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

for seq_id, result in results.items():
    sequence = result["sequence"]
    llr = np.array(result["llr"])
    entropy = np.array(result["entropy"])
    
    # Find all mutations
    mutations = []
    aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
    
    for pos in range(len(sequence)):
        wt_aa = sequence[pos]
        if wt_aa not in aa_to_idx:
            continue
        
        wt_idx = aa_to_idx[wt_aa]
        for mut_idx, mut_aa in enumerate(AMINO_ACIDS):
            if mut_idx != wt_idx:
                mutations.append({
                    'mutation': f"{wt_aa}{pos+1}{mut_aa}",
                    'llr': float(llr[pos, mut_idx]),
                    'entropy': float(entropy[pos])
                })
    
    df_mut = pd.DataFrame(mutations).sort_values('llr', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"{seq_id} - Top Beneficial Mutations:")
    print(f"{'='*60}")
    print(df_mut.head(15)[['mutation', 'llr', 'entropy']].to_string(index=False))
    
    # Save
    df_mut.to_csv(f"{OUTPUT_DIR}/{seq_id}_mutations.csv", index=False)

## Save Results

In [ ]:
for seq_id, result in results.items():
    # Per-position metrics
    df_metrics = pd.DataFrame({
        "position": range(1, len(result["sequence"]) + 1),
        "wildtype_aa": list(result["sequence"]),
        "entropy": result["entropy"],
        "fraction_deleterious": result["fraction_deleterious"]
    })
    df_metrics.to_csv(f"{OUTPUT_DIR}/{seq_id}_metrics.csv", index=False)

print(f"✓ Results saved to {OUTPUT_DIR}/")
print(f"  - Per-position metrics: {{seq_id}}_metrics.csv")
print(f"  - Mutations: {{seq_id}}_mutations.csv")
print(f"  - Analysis plots: {{seq_id}}_analysis.png")

## Interpretation Guide

In [ ]:
display(Markdown("""
## How to Use These Results

### **Design High-Expressing Variants**
- Look for high-entropy (flexible) positions
- Score mutations with positive LLR
- Example: If position 42 has entropy=4.0, try A42V (if LLR > 0)

### **Identify Essential Residues**
- Low entropy positions are critical
- Avoid mutations at positions with entropy < 1.0
- These are likely: catalytic sites, structural cores

### **Predict Mutational Effects**
- **LLR > 0.5**: Likely neutral or beneficial
- **-0.5 < LLR < 0.5**: Likely neutral
- **LLR < -0.5**: Likely deleterious

### **Library Design**
1. Filter for entropy > 2.0 (flexible positions)
2. Score all mutations at those positions
3. Select mutations with LLR > 0 (non-deleterious)
4. Combine into saturation mutagenesis libraries

### **Scaling**
For 100+ proteins, increase Modal `concurrency_limit`:
```python
@app.cls(..., concurrency_limit=5)  # Run 5 in parallel
```
"""))